# 📊 Análise Exploratória de Dados Públicos

Pipeline completo de EDA com dataset socioeconômico sintético baseado em dados públicos.

**Etapas:**
1. Coleta e carregamento
2. Diagnóstico e limpeza
3. Estatística descritiva
4. Visualizações
5. Modelagem preditiva (Random Forest)

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (10, 5)})
os.makedirs('../outputs', exist_ok=True)

print('✅ Bibliotecas carregadas')

---
## 1. Coleta e Geração do Dataset

In [ ]:
from data_processing import gerar_dataset

df_raw = gerar_dataset(n=1000, seed=42)
print(f'Shape bruto: {df_raw.shape}')
df_raw.head()

---
## 2. Diagnóstico e Limpeza

In [ ]:
print('=== Diagnóstico Inicial ===')
print(f'Shape         : {df_raw.shape}')
print(f'Duplicatas    : {df_raw.duplicated().sum()}')
print(f'\nValores nulos:\n{df_raw.isnull().sum()}')

In [ ]:
from data_processing import remover_duplicatas, tratar_nulos, remover_outliers_iqr

df = remover_duplicatas(df_raw)
df = tratar_nulos(df)
df = remover_outliers_iqr(df, 'renda_mensal')

print(f'\nShape final: {df.shape}')

---
## 3. Estatística Descritiva

In [ ]:
from data_processing import estatisticas_descritivas, matriz_correlacao

print('--- Estatísticas Descritivas ---')
display(estatisticas_descritivas(df))

print('\n--- Matriz de Correlação ---')
display(matriz_correlacao(df))

---
## 4. Visualizações

In [ ]:
# Distribuição da Renda Mensal
fig, ax = plt.subplots()
sns.histplot(df['renda_mensal'], bins=40, kde=True, color='#4C72B0', ax=ax)
ax.set_title('Distribuição da Renda Mensal', fontsize=14, fontweight='bold')
ax.set_xlabel('Renda (R$)')
ax.set_ylabel('Frequência')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x:,.0f}'))
plt.tight_layout()
plt.savefig('../outputs/distribuicao_renda.png')
plt.show()

In [ ]:
# Heatmap de Correlação
corr = df.select_dtypes(include=[np.number]).corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Matriz de Correlação', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/heatmap_correlacao.png')
plt.show()

In [ ]:
# Boxplot Renda por Região
fig, ax = plt.subplots()
ordem = df.groupby('regiao')['renda_mensal'].median().sort_values().index
sns.boxplot(data=df, x='regiao', y='renda_mensal', order=ordem, palette='Set2', ax=ax)
ax.set_title('Renda Mensal por Região', fontsize=14, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x:,.0f}'))
plt.tight_layout()
plt.savefig('../outputs/boxplot_renda_regiao.png')
plt.show()

In [ ]:
# Categoria de Renda por Escolaridade
ordem_escol = ['Fundamental', 'Médio', 'Superior', 'Pós-graduação']
ordem_cat = ['Baixa', 'Média', 'Alta']
tab = (
    df.groupby(['escolaridade', 'categoria_renda'], observed=True)
    .size().unstack('categoria_renda')[ordem_cat]
    .loc[ordem_escol]
)
ax = tab.plot(kind='bar', stacked=True, colormap='Blues', figsize=(10, 6))
ax.set_title('Categoria de Renda por Escolaridade', fontsize=14, fontweight='bold')
ax.legend(title='Categoria de Renda', bbox_to_anchor=(1.01, 1))
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig('../outputs/categoria_por_escolaridade.png')
plt.show()

---
## 5. Modelagem Preditiva – Random Forest

In [ ]:
from modeling import preparar_features, treinar_modelo

X, y, encoders, feature_cols = preparar_features(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Treino: {len(X_train)} amostras | Teste: {len(X_test)} amostras')

modelo = treinar_modelo(X_train, y_train)
print('✅ Modelo treinado')

In [ ]:
y_pred = modelo.predict(X_test)
acuracia = accuracy_score(y_test, y_pred)
classes = encoders['categoria_renda'].classes_

print(f'Acurácia: {acuracia:.4f} ({acuracia*100:.1f}%)')
print(f'\nCross-Validation (5-fold): {cross_val_score(modelo, X_train, y_train, cv=5).mean():.4f}')
print('\nRelatório de Classificação:')
print(classification_report(y_test, y_pred, target_names=classes))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes, ax=ax)
ax.set_title('Matriz de Confusão – Random Forest', fontsize=13, fontweight='bold')
ax.set_xlabel('Predição')
ax.set_ylabel('Real')
plt.tight_layout()
plt.savefig('../outputs/confusion_matrix.png')
plt.show()

In [ ]:
# Feature Importance
importancias = pd.Series(modelo.feature_importances_, index=feature_cols).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=importancias.values, y=importancias.index, palette='Blues_r', ax=ax)
ax.set_title('Importância das Features – Random Forest', fontsize=13, fontweight='bold')
ax.set_xlabel('Importância Média (Gini)')
plt.tight_layout()
plt.savefig('../outputs/feature_importance.png')
plt.show()